# Model comparison and results

This notebook compares the performance and size of all CIFAR-10 models trained in previous notebooks:
- **01-DNN**: Fully connected deep neural network
- **02-CNN**: Convolutional neural network (grayscale)
- **03-RGB-CNN**: Convolutional neural network (RGB)
- **04-optimized-CNN**: Hyperparameter-optimized CNN
- **05-augmented-CNN**: CNN trained with data augmentation

## Notebook setup

### Imports

In [1]:
# Standard library imports
import json
import pickle
import time

# Third party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from torchvision import datasets, transforms

# Package imports
import image_classification_tools.pytorch.plotting as plots

# Local imports
import configuration as config

## 1. Load model metadata

In [2]:
# Load model configurations from JSON
with open(config.MODELS_DIR / 'models_config.json', 'r') as f:
    models_metadata = json.load(f)['models']

print(f'Loaded metadata for {len(models_metadata)} models')

Loaded metadata for 6 models


### Prepare test samples

In [3]:
# Prepare test samples for cold start testing
grayscale_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
rgb_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

grayscale_dataset = datasets.CIFAR10(root=config.DATA_DIR, train=False, transform=grayscale_transform)
rgb_dataset = datasets.CIFAR10(root=config.DATA_DIR, train=False, transform=rgb_transform)

grayscale_sample, _ = grayscale_dataset[0]
rgb_sample, _ = rgb_dataset[0]

In [ ]:
cold_start_times = {}

for model_info in models_metadata:

    model_name = model_info['name']
    model_path = config.MODELS_DIR / model_info['model_file']
    
    if not model_path.exists():
        print(f'Skipping {model_name} - model file not found')
        continue
    
    # Select appropriate test sample
    sample = grayscale_sample if model_info['input_type'] == 'grayscale' else rgb_sample
    sample_batch = sample.unsqueeze(0)
    
    # Measure cold start time (average over 5 runs)
    times = []

    for _ in range(5):

        start_time = time.time()
        
        # Load model from disk
        model = torch.load(model_path, map_location=config.DEVICE)
        model.to(config.DEVICE)
        model.eval()
        
        # Make single prediction
        with torch.no_grad():
            _ = model(sample_batch.to(config.DEVICE))
        
        times.append(time.time() - start_time)
        
        # Clean up
        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    cold_start_times[model_name] = np.mean(times)
    print(f'{model_name}: {cold_start_times[model_name]*1000:.2f} ms')

print(f'\nCold start measurement complete for {len(cold_start_times)} models')

## 2. Measure cold start times

In [ ]:
results = []

for model_info in models_metadata:
    model_name = model_info['name']
    results_path = config.RESULTS_DIR / model_info['results_file']
    
    if not results_path.exists():
        print(f"Skipping {model_name} - results file not found")
        continue
    
    # Load saved test results
    with open(results_path, 'rb') as f:
        data = pickle.load(f)
    
    # Calculate model size (4 bytes per float32 parameter)
    model_size_mb = (data['total_params'] * 4) / (1024 ** 2)
    
    # Compile results
    results.append({
        'Model': model_name,
        'Description': model_info['description'],
        'Total Parameters': data['total_params'],
        'Trainable Parameters': data['trainable_params'],
        'Size (MB)': model_size_mb,
        'Test Accuracy (%)': data['test_accuracy'],
        'Cold Start (ms)': cold_start_times.get(model_name, 0) * 1000,
        'true_labels': np.array(data['true_labels']),
        'predictions': np.array(data['predictions']),
        'all_probs': np.array(data['all_probs'])
    })

print(f"Loaded results for {len(results)} models")

## 3. Load test results

In [ ]:
# Create summary dataframe
summary_df = pd.DataFrame([
    {
        'Model': r['Model'],
        'Description': r['Description'],
        'Parameters': f"{r['Total Parameters']:,}",
        'Size (MB)': f"{r['Size (MB)']:.2f}",
        'Accuracy (%)': f"{r['Test Accuracy (%)']:.2f}",
        'Cold Start (ms)': f"{r['Cold Start (ms)']:.2f}"
    }
    for r in results
])

summary_df

## 4. Results summary

## 5. Performance visualizations

### Model performance comparison

In [ ]:
# Sort by accuracy for better visualization
sorted_results = sorted(results, key=lambda x: x['Test Accuracy (%)'])

# Create bar chart
fig, ax = plt.subplots(figsize=(12, 6))

models = [r['Model'] for r in sorted_results]
accuracies = [r['Test Accuracy (%)'] for r in sorted_results]

bars = ax.barh(models, accuracies, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])

# Add value labels on bars
for i, (bar, acc) in enumerate(zip(bars, accuracies)):
    ax.text(acc + 0.5, i, f'{acc:.2f}%', va='center', fontweight='bold')

ax.set_xlabel('Test Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold', pad=20)
ax.set_xlim(0, 100)
ax.grid(axis='x', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

### Efficiency metric

In [ ]:
# Calculate efficiency metric
efficiency_results = []
for r in results:
    params_millions = r['Total Parameters'] / 1e6
    efficiency = r['Test Accuracy (%)'] / params_millions
    efficiency_results.append({
        'Model': r['Model'],
        'Accuracy/Param (M)': efficiency
    })

# Sort by efficiency
efficiency_results = sorted(efficiency_results, key=lambda x: x['Accuracy/Param (M)'], reverse=True)

# Create bar chart
fig, ax = plt.subplots(figsize=(12, 6))

models = [r['Model'] for r in efficiency_results]
efficiencies = [r['Accuracy/Param (M)'] for r in efficiency_results]

bars = ax.barh(models, efficiencies, color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#9b59b6'])

# Add value labels
for i, (bar, eff) in enumerate(zip(bars, efficiencies)):
    ax.text(eff + 0.5, i, f'{eff:.2f}', va='center', fontweight='bold')

ax.set_xlabel('Accuracy per Million Parameters', fontsize=12, fontweight='bold')
ax.set_title('Model Efficiency Comparison', fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

# Print efficiency table
print("\nEfficiency Rankings (Accuracy per Million Parameters):")
print("="*60)
for i, r in enumerate(efficiency_results, 1):
    print(f"{i}. {r['Model']}: {r['Accuracy/Param (M)']:.2f}")
print("="*60)

## 6. Per-class performance analysis

In [ ]:
# Calculate per-class accuracy for each model
class_accuracies = {}

for r in results:
    true = r['true_labels']
    pred = r['predictions']
    
    # Calculate accuracy for each class
    class_acc = []
    for i in range(10):
        mask = (true == i)
        if mask.sum() > 0:
            acc = (pred[mask] == i).sum() / mask.sum() * 100
            class_acc.append(acc)
        else:
            class_acc.append(0)
    
    class_accuracies[r['Model']] = class_acc

# Create dataframe
class_acc_df = pd.DataFrame(class_accuracies, index=config.CLASS_NAMES)

In [ ]:
# Create heatmap
fig, ax = plt.subplots(figsize=(12, 6))

im = ax.imshow(class_acc_df.T, cmap='RdYlGn', aspect='auto', vmin=0, vmax=100)

# Set ticks and labels
ax.set_xticks(np.arange(len(config.CLASS_NAMES)))
ax.set_yticks(np.arange(len(results)))
ax.set_xticklabels(config.CLASS_NAMES, rotation=45, ha='right')
ax.set_yticklabels([r['Model'] for r in results])

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Accuracy (%)', rotation=270, labelpad=20, fontweight='bold')

# Add text annotations
for i in range(len(results)):
    for j in range(len(config.CLASS_NAMES)):
        ax.text(j, i, f'{class_acc_df.iloc[j, i]:.1f}',
                ha="center", va="center", color="black", fontsize=8)

ax.set_title('Per-class accuracy comparison', fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('CIFAR-10 class', fontsize=12, fontweight='bold')
ax.set_ylabel('Model', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nPer-class accuracy summary (%):")
print("="*80)
print(class_acc_df.round(1))
print("="*80)

## 7. Confusion matrix comparison

In [ ]:
# Find best and worst models by accuracy
sorted_by_acc = sorted(results, key=lambda x: x['Test Accuracy (%)'])
worst_model = sorted_by_acc[0]
best_model = sorted_by_acc[-1]

# Plot confusion matrix for worst model
print(f"Worst Model: {worst_model['Model']} - Accuracy: {worst_model['Test Accuracy (%)']:.2f}%")
fig1, ax1 = plots.plot_confusion_matrix(
    worst_model['true_labels'], 
    worst_model['predictions'], 
    config.CLASS_NAMES,
    figsize=(8, 8)
)
ax1.set_title(f'{worst_model["Model"]} - Accuracy: {worst_model["Test Accuracy (%)"]:.2f}%', 
              fontsize=12, fontweight='bold', pad=15)
plt.show()

# Plot confusion matrix for best model
print(f"\nBest Model: {best_model['Model']} - Accuracy: {best_model['Test Accuracy (%)']:.2f}%")
fig2, ax2 = plots.plot_confusion_matrix(
    best_model['true_labels'], 
    best_model['predictions'], 
    config.CLASS_NAMES,
    figsize=(8, 8)
)
ax2.set_title(f'{best_model["Model"]} - Accuracy: {best_model["Test Accuracy (%)"]:.2f}%', 
              fontsize=12, fontweight='bold', pad=15)
plt.show()
